<a href="https://colab.research.google.com/github/hiroaki-com/colab-ollama-private-chat/blob/main/ollama_colab_private_chat_ja.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ollama Colab Private Chat

Google Colab の GPU を活用し、セキュアな環境でローカル LLM とチャットできるノートブックです。

特徴：
- ✅ 完全無料 · 外部 API 非依存 · Colab インスタンス内推論によるプライバシー保護
- 🔒 ログを一切保存しないステートレス設計（ブラウザリロードで即時消去）
- 🚀 環境構築不要・同一ノートブック内でサーバー起動からUI描画まで完結

使い方：
1. `Model Registry` で使用したいモデルを選択してセルを実行
2. `Server` を実行（初回モデルのダウンロードに数分かかります）
3. `Chat UI — Inline`（セル内）または `Chat UI — Standalone`（別タブ）を実行してチャット開始

> - `Inline モード`: Colab の出力セル内で動作。高速で安定した `Direct`（Colab 内部通信）と `Tunnel（Backup）` をシームレスに切替可能
> - `Standalone モード`: ブラウザの別タブで動作する独立 UI。専用 URL が発行され、スマホなど他の端末からもアクセス可能

In [ ]:
#@title 📋 Model Registry

# @markdown ### モデル設定
# @markdown 追加したいモデル名をカンマ区切りで入力して下さい。
# @markdown > 正式なモデル名はこちらから：https://ollama.com/search

model_list = "nemotron-3-nano:4b, ministral-3:3b, qwen3.5:4b" #@param {type:"string"}
num_ctx = 8192 #@param {type:"integer"}

AVAILABLE_MODELS = [
    model.strip()
    for model in model_list.split(',')
    if model.strip()
]

if not AVAILABLE_MODELS:
    raise ValueError("❌ モデルリストが空です。少なくとも1つのモデルを入力してください。")

import ipywidgets as widgets
from IPython.display import display

header = widgets.HTML(
    '<h3>📦 モデルを選択</h3>'
    '<p style="margin: 5px 0 10px 0; font-size: 13px;">'
    '起動するモデルを1つ選択し、次のセルを実行してください。</p>'
)

model_selector = widgets.RadioButtons(
    options=AVAILABLE_MODELS,
    value=AVAILABLE_MODELS[0],
    layout=widgets.Layout(margin='0 0 0 20px')
)

display(widgets.VBox([header, model_selector]))

effective_ctx = num_ctx if num_ctx > 0 else None

print(f"✅ モデルリストを読み込みました: {len(AVAILABLE_MODELS)}個")
print("➡️ モデルを選択後、次のセル（Server）を実行してください。")

In [ ]:
#@title 🚀 Ollama Server

# @markdown Cloudflare Tunnel を起動し、外部アクセス用の公開 URL を発行します。
# @markdown
# @markdown Web Search（RAG）設定
# @markdown
# @markdown | パラメータ | 説明 |
# @markdown |---|---|
# @markdown | `SEARCH_MAX_RESULTS` | 取得する検索結果の最大件数 |
# @markdown | `SEARCH_BODY_LENGTH` | 各検索結果から切り出す本文の最大文字数 |
# @markdown | `SEARCH_TIME_LIMIT` | 検索対象の期間フィルター |
# @markdown | `SEARCH_REGION` | 検索対象の地域・言語 |


SEARCH_MAX_RESULTS = 5   #@param {type:"integer"}
SEARCH_BODY_LENGTH = 300  #@param {type:"integer"}
SEARCH_TIME_LIMIT = "制限なし" #@param ["制限なし", "当日", "1週間", "1ヶ月", "1年"]
SEARCH_REGION     = "日本語"  #@param ["日本語", "全世界", "英語（米国）", "英語（英国）", "中国語（簡体字）", "韓国語", "ドイツ語", "フランス語"]

_time_map   = {"制限なし": None, "当日": "d", "1週間": "w", "1ヶ月": "m", "1年": "y"}
_region_map = {"日本語": "jp-jp", "全世界": "wt-wt", "英語（米国）": "us-en", "英語（英国）": "uk-en", "中国語（簡体字）": "cn-zh", "韓国語": "kr-kr", "ドイツ語": "de-de", "フランス語": "fr-fr"}
_search_timelimit = _time_map[SEARCH_TIME_LIMIT]
SEARCH_REGION     = _region_map[SEARCH_REGION]

MAX_HEALTH_RETRIES   = 150
HEALTH_CHECK_TIMEOUT = 2

BLUE  = "\033[34m"
GREEN = "\033[32m"
GRAY  = "\033[90m"
RESET = "\033[0m"

selected_model = model_selector.value

print(f"\nOLLAMA COLAB SERVER 🚀")
print(f"{GRAY}──────────────────────────────────────────{RESET}")

print(f"  {BLUE}◦{RESET} System   依存パッケージのインストール")
!which zstd > /dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq zstd > /dev/null)
!which ollama > /dev/null 2>&1 || curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!which cloudflared > /dev/null 2>&1 || (wget -qO /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared)

import re, subprocess, sys, time, os, requests, threading, warnings, json, base64

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ddgs"], capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "cachetools"], capture_output=True)

if not re.fullmatch(r'[a-zA-Z0-9._:/-]+', selected_model):
    raise ValueError(f"モデル名が不正です: {selected_model}")

# 環境変数の設定（CORS許可とホスト固定）
os.environ['OLLAMA_HOST']              = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS']           = '*'
os.environ['OLLAMA_KEEP_ALIVE']        = '24h'
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '1'
os.environ['OLLAMA_FLASH_ATTENTION']   = '1'
os.environ['OLLAMA_KV_CACHE_TYPE']     = 'q8_0'

!pkill ollama || true
subprocess.Popen(["/usr/local/bin/ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=os.environ)

for _ in range(MAX_HEALTH_RETRIES):
    try:
        if requests.get("http://0.0.0.0:11434/api/tags", timeout=HEALTH_CHECK_TIMEOUT).status_code == 200:
            print(f"  {GREEN}✓{RESET} Ready    Ollama サーバー起動完了")
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(0.4)
else:
    raise RuntimeError("⚠️ 起動確認に失敗しました。")

print(f"  {BLUE}◦{RESET} Network  Cloudflare Tunnel 確立中")
cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:11434'],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE
)
public_url = None
for line in iter(cf_proc.stderr.readline, b''):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line.decode())
    if m:
        public_url = m.group(0)
        break
if not public_url:
    raise RuntimeError("⚠️ Cloudflare Tunnel の URL 取得に失敗しました。")
print(f"  {GREEN}✓{RESET} Public   {public_url}")

print(f"\n  ▶ Model    {selected_model}")
subprocess.run(["/usr/local/bin/ollama", "pull", selected_model], env=os.environ, check=True)

# Warmup and VRAM monitor
print(f"  {BLUE}◦{RESET} VRAM     モデルをVRAMにロード中...", end="", flush=True)
warmup_payload = {"model": selected_model, "prompt": "hi", "stream": False}
if effective_ctx: warmup_payload["options"] = {"num_ctx": effective_ctx}

def check_vram():
    try:
        ps = requests.get("http://0.0.0.0:11434/api/ps").json()
        base = selected_model.split(':')[0]
        return any(m['name'].split(':')[0] == base for m in ps.get('models', []))
    except Exception: return False

MAX_VRAM_WAIT = 90
warmup_thread = threading.Thread(
    target=lambda: requests.post("http://0.0.0.0:11434/api/generate", json=warmup_payload),
    daemon=True
)
warmup_thread.start()

for _ in range(MAX_VRAM_WAIT):
    if not warmup_thread.is_alive() and check_vram():
        print(f"\n  {GREEN}✓{RESET} Loaded   モデルをVRAMにロード済み")
        break
    print(".", end="", flush=True)
    time.sleep(2)
else:
    print(f"\n  ℹ️ サーバー準備完了。モデルはロード継続中のため、次へ進めますが初回応答に時間がかかる場合があります。")

# === Web Search RAG ===
print(f"  {BLUE}◦{RESET} RAG      DuckDuckGo 検索エンジン初期化中")
warnings.filterwarnings("ignore", category=DeprecationWarning)
from cachetools import TTLCache
from ddgs import DDGS
from google.colab import output as _colab_output

_search_cache      = TTLCache(maxsize=20, ttl=300)
_search_cache_lock = threading.Lock()

def _extract_search_query(user_text: str) -> str:
    """ユーザー入力から検索エンジン向けの短いキーワードを抽出する。失敗時は元のテキストを返す。"""
    try:
        payload = {
            "model": selected_model,
            "prompt": (
                "次のユーザー入力から、検索エンジン向けの短いキーワードを1行のみで抽出してください。"
                "説明・前置き・記号は一切不要。キーワードだけを出力:\n" + user_text
            ),
            "stream": False,
            "options": {"num_ctx": 512},
        }
        r = requests.post(
            "http://0.0.0.0:11434/api/generate",
            json=payload,
            timeout=10,
        )
        keyword = r.json().get("response", "").strip()
        return keyword if keyword else user_text
    except Exception:
        return user_text

def _duckduckgo_search(raw_query: str) -> str:
    """
    ユーザー入力から検索キーワードを抽出し DuckDuckGo を検索する。
    結果は base64 エンコードされた JSON 文字列として返す。
    """
    query = _extract_search_query(raw_query)
    q_key = query.strip().lower()

    with _search_cache_lock:
        if q_key in _search_cache:
            return _search_cache[q_key]

    results = []
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            with DDGS() as ddgs:
                for r in (ddgs.text(
                    query=query,
                    region=SEARCH_REGION,
                    safesearch='moderate',
                    timelimit=_search_timelimit,
                    max_results=SEARCH_MAX_RESULTS,
                ) or []):
                    results.append({
                        "title":   r.get("title", "タイトルなし"),
                        "url":     r.get("href", ""),
                        "content": r.get("body", "")[:SEARCH_BODY_LENGTH],
                    })
    except Exception as e:
        print(f"❌ [RAG] DuckDuckGo Search Error: {type(e).__name__}: {e}")
        if "Ratelimit" in str(e) or "202" in str(e):
            print("⚠️ レート制限が発生しました。少し時間を置いて試してください。")
        return base64.b64encode(b"[]").decode("ascii")

    payload = json.dumps(results, ensure_ascii=False)
    encoded = base64.b64encode(payload.encode("utf-8")).decode("ascii")

    with _search_cache_lock:
        _search_cache[q_key] = encoded
    return encoded

_colab_output.register_callback("searxng_search", _duckduckgo_search)
SEARXNG_ENABLED_JS = "true"
print(f"  {GREEN}✓{RESET} RAG      DuckDuckGo Web Search 準備完了")

print(f"\n{GRAY}──────────────────────────────────────────{RESET}")
print(f"STATUS   : {GREEN}RUNNING (Background){RESET}")
print(f"ENDPOINT : {public_url}")
print(f"{GRAY}──────────────────────────────────────────{RESET}")
print("\n✅ 準備が整いました。次のセルを実行してください。")


In [ ]:
#@title 💬 Chat UI — Inline

try:
    _ = SEARXNG_ENABLED_JS
except NameError:
    SEARXNG_ENABLED_JS = "false"

# @markdown `Direct`（推奨）: Colab 内部のカーネル通信を利用し、高速に動作します。
# @markdown
# @markdown `Tunnel (Backup)`（代替）: Cloudflare Tunnel を経由して接続します。Direct 通信が不安定な場合のバックアップとして使用してください。

import base64
import json
import threading
import uuid

import requests
from IPython.display import HTML, display
from google.colab import output

print(f"  {BLUE}◦{RESET} UI       Chat UI を構成中...")

_model  = model_selector.value
_ctx_js = str(effective_ctx) if effective_ctx else "null"
_max_turns = max(4, ((effective_ctx or 4096) * 3 // 4) // 400 * 2)
_max_turns_js = str(_max_turns)

_stream_jobs = {}
_stream_jobs_lock = threading.Lock()

def _stream_start(model, messages, ctx):
    with _stream_jobs_lock:
        done_keys = [k for k, v in _stream_jobs.items() if v['done']]
        for k in done_keys:
            del _stream_jobs[k]
    job_id = uuid.uuid4().hex
    _stream_jobs[job_id] = {'buf': b'', 'done': False, 'error': None, 'cancel': threading.Event()}
    def _run():
        cancel_ev = _stream_jobs[job_id]['cancel']
        try:
            payload = {"model": model, "messages": messages, "stream": True}
            if ctx:
                payload["options"] = {"num_ctx": int(ctx)}
            with requests.post("http://localhost:11434/api/chat", json=payload, timeout=300, stream=True) as r:
                r.raise_for_status()
                for line in r.iter_lines():
                    if cancel_ev.is_set():
                        break
                    if not line:
                        continue
                    try:
                        chunk = json.loads(line).get('message', {}).get('content', '')
                        if not chunk:
                            continue
                        with _stream_jobs_lock:
                            _stream_jobs[job_id]['buf'] += chunk.encode('utf-8')
                    except Exception:
                        pass
        except Exception as e:
            with _stream_jobs_lock:
                _stream_jobs[job_id]['error'] = str(e)
        finally:
            with _stream_jobs_lock:
                _stream_jobs[job_id]['done'] = True
    threading.Thread(target=_run, daemon=True).start()
    return job_id

def _stream_poll(job_id, offset=0):
    job = _stream_jobs.get(job_id)
    if not job:
        return 'ERR|' + base64.b64encode(b'job not found').decode() + '|'
    with _stream_jobs_lock:
        buf_snapshot = job['buf']
        err_snapshot = job['error'] or ''
        is_done      = job['done']
        if is_done:
            _stream_jobs.pop(job_id, None)
    delta_bytes = buf_snapshot[offset:]
    buf_b64 = base64.b64encode(delta_bytes).decode()
    err_b64 = base64.b64encode(err_snapshot.encode()).decode() if err_snapshot else ''
    result = ('DONE' if is_done else 'WAIT') + '|' + buf_b64 + '|' + err_b64
    return result

output.register_callback("ollama_stream_start", _stream_start)
output.register_callback("ollama_stream_poll",  _stream_poll)

def _stream_cancel(job_id):
    with _stream_jobs_lock:
        job = _stream_jobs.get(job_id)
    if job:
        job['cancel'].set()

output.register_callback("ollama_stream_cancel", _stream_cancel)

_INLINE_UI_HTML = r"""<link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;600&family=IBM+Plex+Sans+JP:wght@400;500&display=swap" rel="stylesheet">
<script src="https://cdnjs.cloudflare.com/ajax/libs/marked/9.1.6/marked.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/dompurify/3.0.8/purify.min.js"></script>
<style>
:root{--bg:#f5f0e8;--surface:#faf7f2;--border:#c8bfae;--accent:#3a6b4a;--text:#2c2820;--muted:#8a7f72;--user-bg:#e8edf5;--ai-bg:#f0f5ee}
*{box-sizing:border-box;margin:0;padding:0}
#oc{font-family:'IBM Plex Sans JP',sans-serif;background:var(--bg);color:var(--text);max-width:700px;padding:14px}
.oc-ph{font-family:'IBM Plex Mono',monospace;display:flex;justify-content:space-between;align-items:baseline;padding:8px 0 12px;border-bottom:1px solid var(--border);margin-bottom:12px}
.oc-ph .t{font-size:13px;font-weight:600;letter-spacing:.04em}
.oc-ph .s{font-size:11px;color:var(--muted)}
.oc-tabs{display:flex;gap:4px;margin-bottom:10px}
.oc-tab{font-family:'IBM Plex Mono',monospace;font-size:12px;padding:4px 14px;border:1px solid var(--border);background:var(--surface);color:var(--muted);cursor:pointer}
.oc-tab.on{background:var(--accent);color:#fff;border-color:var(--accent)}
.oc-sb{display:flex;align-items:center;gap:8px;margin-bottom:10px;font-size:12px;font-family:'IBM Plex Mono',monospace;color:var(--muted)}
.oc-sb input{flex:1;padding:5px 8px;border:1px solid var(--border);background:var(--surface);color:var(--text);font-family:'IBM Plex Mono',monospace;font-size:12px;outline:none}
.oc-card{border:1px solid var(--border);background:var(--surface)}
.oc-ch{display:flex;justify-content:space-between;align-items:center;padding:8px 12px;border-bottom:1px solid var(--border);font-size:12px;font-family:'IBM Plex Mono',monospace}
.led{display:inline-block;width:8px;height:8px;border-radius:50%;background:#bbb;margin-right:6px;vertical-align:middle}
.led.on{background:var(--accent);animation:p 1.2s ease-in-out infinite}
@keyframes p{0%,100%{opacity:1}50%{opacity:.3}}
.tag{display:inline-block;padding:1px 6px;border:1px solid var(--border);font-size:11px;color:var(--muted);margin-left:4px}
.oc-log{height:340px;overflow-y:auto;padding:12px;display:flex;flex-direction:column;gap:8px}
.sep{text-align:center;font-size:11px;color:var(--muted);font-family:'IBM Plex Mono',monospace;padding:4px 0;border-top:1px dashed var(--border);margin:4px 0}
.bbl{max-width:85%;padding:8px 12px;font-size:13px;line-height:1.65;word-break:break-word}
.bbl.u{align-self:flex-end;background:var(--user-bg);border:1px solid #c5d0e0}
.bbl.a{align-self:flex-start}
.bbl.e{align-self:flex-start;background:var(--ai-bg);border:1px solid #e0b8b8;color:#c0392b}
.bbl pre{background:#e8e4dc;padding:8px;overflow-x:auto;font-family:'IBM Plex Mono',monospace;font-size:12px;margin-top:6px;white-space:pre-wrap}
.tk{display:flex;gap:4px;padding:6px 0;align-items:center;min-height:20px}
.tk div{width:7px;height:7px;background:var(--accent);border-radius:50%;animation:tk 1.4s infinite ease-in-out;opacity:.4}
.tk div:nth-child(2){animation-delay:.2s}
.tk div:nth-child(3){animation-delay:.4s}
@keyframes tk{0%,80%,100%{transform:scale(0.8);opacity:.4}40%{transform:scale(1.2);opacity:1}}
.oc-ia{display:flex;gap:8px;padding:10px 12px;border-top:1px solid var(--border)}
.oc-ia textarea{flex:1;padding:7px 10px;border:1px solid var(--border);background:var(--bg);color:var(--text);font-family:'IBM Plex Sans JP',sans-serif;font-size:13px;resize:none;min-height:38px;max-height:120px;line-height:1.5;outline:none}
.oc-ia textarea::placeholder{font-size:clamp(10px,2.2vw,13px)}
.oc-ia button{padding:7px 18px;background:var(--accent);color:#fff;border:none;font-size:13px;cursor:pointer;font-family:'IBM Plex Mono',monospace;flex-shrink:0}
.oc-ia button:disabled,.oc-ia textarea:disabled{opacity:.5;cursor:not-allowed}
#ocAbortBtn{background:none;border:1px solid var(--border);color:var(--muted)}
#ocAbortBtn:hover{border-color:#c0392b;color:#c0392b}
.oc-cf{display:flex;justify-content:space-between;align-items:center;padding:6px 12px;border-top:1px solid var(--border);font-size:11px;font-family:'IBM Plex Mono',monospace;color:var(--muted)}
.oc-cf button{background:none;border:1px solid var(--border);padding:2px 8px;font-size:11px;color:var(--muted);cursor:pointer}
.oc-note{margin-top:10px;font-size:11px;color:var(--muted);font-family:'IBM Plex Mono',monospace;border:1px dashed var(--border);padding:8px 12px;line-height:1.8}
.bbl p{margin-bottom:8px}.bbl p:last-child{margin-bottom:0}.bbl h1,.bbl h2,.bbl h3{font-weight:600;margin:10px 0 4px;color:inherit}.bbl ul,.bbl ol{margin:6px 0 6px 18px}.bbl li{margin-bottom:2px}.bbl code{background:#e8e4dc;padding:1px 4px;font-family:'IBM Plex Mono',monospace;font-size:12px}.bbl blockquote{border-left:3px solid var(--border);padding-left:8px;color:var(--muted);margin:6px 0}.sep{cursor:pointer}.sep:hover{color:var(--accent)}.u-turn{display:flex;flex-direction:column;align-self:flex-end;gap:2px}.u-wrap{display:flex;align-items:flex-start;gap:6px}.u-foot{display:flex;justify-content:flex-end}.cp-btn{display:inline-block;margin:0;padding:3px 7px;font-size:13px;background:none;border:1px solid transparent;border-radius:6px;color:var(--muted);cursor:pointer;font-family:'IBM Plex Mono',monospace;opacity:0;transition:opacity .15s}.cp-btn:hover{background:rgba(0,0,0,.06);border-color:rgba(0,0,0,.15);color:var(--muted)}.bbl:hover .cp-btn{opacity:1}.u-turn:hover .cp-btn{opacity:1}
.edit-ta{width:100%;min-height:60px;max-height:200px;padding:6px 8px;font-family:'IBM Plex Sans JP',sans-serif;font-size:13px;background:var(--bg);color:var(--text);border:1px solid var(--accent);resize:vertical;outline:none;box-sizing:border-box}
#ocWebToggle{display:inline-flex;align-items:center;gap:6px;background:none;border:1px solid var(--border);padding:2px 8px 2px 6px;font-size:11px;color:var(--muted);cursor:pointer;font-family:'IBM Plex Mono',monospace;border-radius:10px;transition:background .15s,color .15s,border-color .15s}
#ocWebToggle.on{border-color:var(--accent);color:var(--accent)}
#ocWebToggle .sw-track{display:inline-block;width:24px;height:13px;border-radius:7px;background:var(--border);position:relative;flex-shrink:0;transition:background .15s}
#ocWebToggle .sw-track::after{content:'';position:absolute;top:2px;left:2px;width:9px;height:9px;border-radius:50%;background:#fff;transition:transform .15s}
#ocWebToggle.on .sw-track{background:var(--accent)}
#ocWebToggle.on .sw-track::after{transform:translateX(11px)}
.src-list{font-size:10px;color:var(--muted);font-family:'IBM Plex Mono',monospace;margin-top:4px;padding:4px 8px;border-left:2px solid var(--border);line-height:1.7}
.src-list a{color:var(--accent);text-decoration:none}
.src-list a:hover{color:var(--accent)}
</style>
<div id="oc">
  <div class="oc-ph">
    <span class="t">OLLAMA COLAB PRIVATE CHAT</span>
    <span class="s">no history · local inference</span>
  </div>
  <div class="oc-tabs">
    <button id="tabInline" class="oc-tab on" onclick="ocMode('inline')">[ Direct ]</button>
    <button id="tabTunnel" class="oc-tab" onclick="ocMode('tunnel')">[ Tunnel (Backup) ]</button>
  </div>
  <div class="oc-sb">
    BASE URL
    <input id="ocUrl" type="text" readonly>
  </div>
  <div class="oc-card">
    <div class="oc-ch">
      <span><span id="ocLed" class="led"></span></span>
      <span><span class="tag" id="ocMTag"></span></span>
    </div>
    <div id="ocLog" class="oc-log"></div>
    <div class="oc-ia">
      <textarea id="ocIn" rows="1" placeholder="何でも聞いてみよう"></textarea>
      <button id="ocBtn" onclick="ocSend()">送信</button>
      <button id="ocAbortBtn" onclick="ocAbort()" style="display:none">中断</button>
    </div>
    <div class="oc-cf">
      <span>stateless · no localStorage</span>
      <div style="display:flex;gap:4px">
      <button id="ocWebToggle" title="Web検索RAGのON/OFF"><span class="sw-track"></span>🔍 Web検索</button>
      <button onclick="ocClear()">ログ消去</button>
    </div>
    </div>
  </div>
  <div id="ocNt" class="oc-note">
    ℹ️ Direct はデフォルトです。Colab 内部通信を使用します。Direct が不安定な場合や検証時は Tunnel (Backup) を選択してください。
  </div>
</div>
<script>
(function(){
const MODEL="__MODEL__", CTX=__CTX_JS__, MAX_TURNS=__MAX_TURNS__, TUNNEL_URL="__TUNNEL_URL__";
let currentMode = 'inline';
const $=id=>document.getElementById(id);
const elLog=$('ocLog'), elIn=$('ocIn'), elBtn=$('ocBtn'), elLed=$('ocLed'), elNt=$('ocNt'), elUrl=$('ocUrl');
const elAbortBtn = $('ocAbortBtn');
let _abort = null;
let _history = [];
window.ocAbort = function() { if (_abort) _abort(); };
const SEARXNG_ENABLED = __SEARXNG_ENABLED__;
let _webSearch = false;
const elWebToggle = $('ocWebToggle');
if (!SEARXNG_ENABLED) {
  elWebToggle.style.display = 'none';
} else {
  elWebToggle.onclick = function() {
    _webSearch = !_webSearch;
    elWebToggle.classList.toggle('on', _webSearch);
  };
}

let _lastSearchFailed = false;
async function fetchSearchResults(query) {
  _lastSearchFailed = false;
  try {
    const raw = await google.colab.kernel.invokeFunction('searxng_search', [query], {});
    const b64 = raw.data['text/plain'].slice(1, -1);
    const bytes = Uint8Array.from(atob(b64), c => c.charCodeAt(0));
    return JSON.parse(new TextDecoder().decode(bytes));
  } catch(e) {
    console.warn('[RAG] fetchSearchResults failed:', e);
    _lastSearchFailed = true;
    return [];
  }
}
function renderSources(sources, targetEl) {
  if (!sources || !sources.length) return;
  const d = document.createElement('div');
  d.className = 'src-list';
  d.innerHTML = sources.map((s, i) =>
    `<sup><a href="${s.url}" target="_blank" rel="noopener" title="${s.url}">[${i+1}]</a></sup> ${s.title || s.url}`
  ).join('<br>');
  targetEl.appendChild(d);
}

$('ocMTag').textContent = MODEL;
elUrl.value = '(Direct: Python kernel)';

window.ocMode = function(m) {
  currentMode = m;
  $('tabInline').classList.toggle('on', m==='inline');
  $('tabTunnel').classList.toggle('on', m==='tunnel');
  elUrl.value = m === 'inline' ? '(Direct: Python kernel)' : TUNNEL_URL;
  if (m === 'inline') {
    elNt.innerHTML = 'ℹ️ Direct はデフォルトです。Colab 内部通信を使用します。Direct が不安定な場合や検証時は Tunnel (Backup) を選択してください。';
  } else {
    elNt.innerHTML = '⚠️ Tunnel (Backup) モード: Cell 2 を再実行した場合は Cell 3 も再実行してください（URL が変わるため）。';
  }
  elNt.style.display = '';
};

window.ocClear = function() { elLog.innerHTML = ''; _history = []; };
function ledOn(v) { elLed.className = 'led' + (v ? ' on' : ''); }
function md(s) { return DOMPurify.sanitize(marked.parse(s)); }
function addSep() {
  const d = document.createElement('div');
  d.className = 'sep';
  d.textContent = '↻ リセット';
  d.onclick = function(){
    while (elLog.firstChild && elLog.firstChild !== d)
      elLog.removeChild(elLog.firstChild);
    d.remove();
  };
  elLog.appendChild(d);
}
function addBubble(cls, content, asHtml) {
  const d = document.createElement('div'); d.className = 'bbl ' + cls;
  if (asHtml) d.innerHTML = content; else d.textContent = content;
  elLog.appendChild(d); return d;
}
function setDisabled(v) {
  elBtn.disabled = v;
  elAbortBtn.style.display = v ? '' : 'none';
}

window.ocSend = async function() {
  const t = elIn.value.trim();
  if (!t) return;
  elIn.value = ''; elIn.style.height = '';
  const historyLenBefore = _history.length;
  _history.push({ role: 'user', content: t });

  let _sources = [];
  let _messagesForLLM = _history.slice(-MAX_TURNS);
  const _todayStr = new Date().toLocaleDateString('ja-JP', {year:'numeric',month:'long',day:'numeric',weekday:'long'});
  let _sysContent = `今日の日付: ${_todayStr}`;

  if (_webSearch) {
    const _searchInd = addBubble('a', '🔍 Web検索中…', false);
    elLog.scrollTop = elLog.scrollHeight;
    _sources = await fetchSearchResults(t);
    _searchInd.remove();
    if (_lastSearchFailed) {
      addBubble('a', '⚠️ Web検索に失敗しました。LLM単体で回答します。', false);
    }
    if (_sources.length) {
      const contextText = _sources.map((s, i) =>
        `[${i+1}] ${s.title}\n${s.content}`
      ).join('\n\n');
      _sysContent += `\n\nあなたは、提供されたWeb検索結果をもとに質問に回答するアシスタントです。\n\n【ルール】\n- 回答は検索結果を優先すること。引用する場合は出典番号 [n] を付けること。\n- 検索結果にない情報は「検索結果には含まれていません」と明示した上で、自身の知識で補足してよい。\n\n【検索結果】\n${contextText}`;
    }
  }
  _messagesForLLM = [{ role: 'system', content: _sysContent }, ..._messagesForLLM];

  if (elLog.children.length) addSep();
  const turnDiv = document.createElement('div');
  turnDiv.className = 'u-turn';
  const retryBtn = document.createElement('button');
  retryBtn.className = 'cp-btn'; retryBtn.textContent = '↻'; retryBtn.title = 'リトライ';
  function doRetry(newText) {
    if (elBtn.disabled) return;
    const prev = turnDiv.previousSibling;
    if (prev && prev.classList.contains('sep')) prev.remove();
    while (turnDiv.nextSibling) elLog.removeChild(turnDiv.nextSibling);
    _history.length = historyLenBefore;
    turnDiv.remove();
    elIn.value = newText;
    ocSend();
  }
  retryBtn.onclick = function() { doRetry(t); };
  const userBubble = document.createElement('div');
  userBubble.className = 'bbl u';
  userBubble.textContent = t;
  const wrapDiv = document.createElement('div');
  wrapDiv.className = 'u-wrap';
  wrapDiv.appendChild(retryBtn);
  wrapDiv.appendChild(userBubble);
  const foot = document.createElement('div');
  foot.className = 'u-foot';
  const copyUserBtn = document.createElement('button');
  copyUserBtn.className = 'cp-btn'; copyUserBtn.textContent = '⧉'; copyUserBtn.title = 'コピー';
  copyUserBtn.onclick = function() { navigator.clipboard.writeText(t).then(function(){ copyUserBtn.textContent = '✓'; setTimeout(function(){ copyUserBtn.textContent = '⧉'; }, 1500); }); };
  foot.appendChild(copyUserBtn);

  const editBtn = document.createElement('button');
  editBtn.className = 'cp-btn'; editBtn.textContent = '✎'; editBtn.title = '編集';
  editBtn.onclick = enterEditMode;
  foot.appendChild(editBtn);

  function enterEditMode() {
    const ta = document.createElement('textarea');
    ta.className = 'edit-ta';
    ta.value = t;
    userBubble.replaceWith(ta);
    ta.focus(); ta.select();

    copyUserBtn.style.display = 'none';
    editBtn.style.display = 'none';

    const confirmBtn = document.createElement('button');
    confirmBtn.className = 'cp-btn'; confirmBtn.textContent = '✓'; confirmBtn.title = '確定';
    const cancelBtn = document.createElement('button');
    cancelBtn.className = 'cp-btn'; cancelBtn.textContent = '✕'; cancelBtn.title = 'キャンセル';

    function exitEditMode() {
      ta.replaceWith(userBubble);
      copyUserBtn.style.display = '';
      editBtn.style.display = '';
      confirmBtn.remove();
      cancelBtn.remove();
    }

    confirmBtn.onclick = function() {
      const newText = ta.value.trim();
      if (newText) doRetry(newText);
      else exitEditMode();
    };
    cancelBtn.onclick = exitEditMode;

    ta.addEventListener('keydown', function(e) {
      if (e.key === 'Enter' && (e.ctrlKey || e.metaKey)) { e.preventDefault(); confirmBtn.onclick(); }
    });

    foot.appendChild(confirmBtn);
    foot.appendChild(cancelBtn);
  }
  turnDiv.appendChild(wrapDiv);
  turnDiv.appendChild(foot);
  elLog.appendChild(turnDiv);
  elLog.scrollTop = elLog.scrollHeight;

  const ai = addBubble('a', '<div class="tk"><div></div><div></div><div></div></div>', true);
  elLog.scrollTop = elLog.scrollHeight;
  setDisabled(true); ledOn(true);

  let full = '';
  try {
    let _rafId = null;
    if (currentMode === 'inline') {
      const startRaw = await google.colab.kernel.invokeFunction('ollama_stream_start', [MODEL, _messagesForLLM, CTX], {});
      const jobId = startRaw.data['text/plain'].slice(1, -1);
      let firstChunk = true;
      let _abortA = false, offset = 0;
      _abort = () => { _abortA = true; google.colab.kernel.invokeFunction('ollama_stream_cancel', [jobId], {}); };
      while (true) {
        if (_abortA) break;
        const pollRaw = await google.colab.kernel.invokeFunction('ollama_stream_poll', [jobId, offset], {});
        const s = pollRaw.data['text/plain'].slice(1, -1);
        const pipe1 = s.indexOf('|'), pipe2 = s.indexOf('|', pipe1 + 1);
        const status = s.slice(0, pipe1);
        const deltaB64 = s.slice(pipe1 + 1, pipe2);
        const errB64 = s.slice(pipe2 + 1);
        if (deltaB64) {
          const deltaBytes = Uint8Array.from(atob(deltaB64), c => c.charCodeAt(0));
          const delta = new TextDecoder().decode(deltaBytes);
          if (delta) {
            full += delta;
            offset += deltaBytes.length;
            if (firstChunk) { ai.innerHTML = ''; firstChunk = false; }
            ai.innerHTML = md(full);
          }
        }
        if (status === 'ERR') {
           const errMsg = errB64 ? new TextDecoder().decode(Uint8Array.from(atob(errB64), c => c.charCodeAt(0))) : 'Streaming Error';
           throw new Error(errMsg);
        }
        if (status === 'DONE') break;
        await new Promise(r => setTimeout(r, 120));
      }
    } else {
      // ── Tunnel (Backup) ──────────────────────────────────────────
      const ctrl = new AbortController();
      _abort = () => ctrl.abort();
      const p = { model: MODEL, messages: _messagesForLLM, stream: true };
      if (CTX) p.options = { num_ctx: parseInt(CTX) };
      const res = await fetch(TUNNEL_URL + '/api/chat', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify(p), signal: ctrl.signal
      });
      if (!res.ok) throw new Error('HTTP ' + res.status);
      const reader = res.body.getReader(), dec = new TextDecoder();
      let buf = '', firstChunk = true;
      while (true) {
        const { done, value } = await reader.read();
        if (done) break;
        buf += dec.decode(value, { stream: true });
        const lines = buf.split('\n');
        buf = lines.pop();
        for (const l of lines) {
          if (!l.trim()) continue;
          try {
            const chunk = JSON.parse(l).message?.content || '';
            if (!chunk) continue;
            if (firstChunk) { ai.innerHTML = ''; firstChunk = false; }
            full += chunk;
            ai.innerHTML = md(full);
          } catch(_) {}
        }
      }
    }
    _history.push({ role: 'assistant', content: full });
    renderSources(_sources, ai);
  } catch (e) {
    _history.length = historyLenBefore;
    if (e.name === 'AbortError') {
      if (full) { ai.innerHTML = md(full); }
      else { ai.innerHTML = ''; ai.textContent = '（中断）'; }
    } else {
      ai.className = 'bbl e';
      ai.textContent = 'エラー: ' + e.message;
    }
  } finally {
    _abort = null;
    setDisabled(false); ledOn(false);
  }
};
elIn.addEventListener('keydown', e => {
  if (e.key === 'Enter' && (e.ctrlKey || e.metaKey)) { e.preventDefault(); ocSend(); return; }
  if (e.key === 'Enter' && !e.shiftKey && !e.isComposing) { e.preventDefault(); ocSend(); }
});
elIn.addEventListener('input', function() { this.style.height = ''; this.style.height = Math.min(this.scrollHeight, 120) + 'px'; });
elIn.focus();

// フォーム外のダブルタップでテキストエリアにフォーカス
let _lastClick = 0;
document.getElementById('oc').addEventListener('click', function(e) {
  let now = Date.now();
  if (now - _lastClick < 300 && !_abort && !e.target.closest('.oc-ia, .cp-btn, .edit-ta, button')) {
    elIn.focus();
  }
  _lastClick = now;
});
})();
</script>
"""

display(HTML(_INLINE_UI_HTML.replace("__MODEL__", _model).replace("__CTX_JS__", _ctx_js).replace("__MAX_TURNS__", _max_turns_js).replace("__TUNNEL_URL__", public_url).replace("__SEARXNG_ENABLED__", SEARXNG_ENABLED_JS)))

In [ ]:
#@title 🌐 Chat UI — Standalone

try:
    _ = SEARXNG_ENABLED_JS
except NameError:
    SEARXNG_ENABLED_JS = "false"

# @markdown ブラウザの別タブ全体を使ってチャットができる独立した UI です。
# @markdown
# @markdown 発行される Cloudflare Tunnel の URL にアクセスして使用します（他の端末への共有も可能）。

print(f"  {BLUE}◦{RESET} System   依存パッケージを確認中")
!pip install -q fastapi uvicorn httpx
print(f"  {GREEN}✓{RESET} System   パッケージ確認完了")

!pkill -f "uvicorn.*11435"     > /dev/null 2>&1 || true
!pkill -f "cloudflared.*11435" > /dev/null 2>&1 || true

import asyncio
import re
import subprocess
import threading
import time

import httpx
import requests
import uvicorn
from cachetools import TTLCache
from ddgs import DDGS
from fastapi import FastAPI, Request
from fastapi.responses import HTMLResponse, StreamingResponse
from IPython.display import HTML, display

time.sleep(1)

_model  = model_selector.value
_ctx_js = str(effective_ctx) if effective_ctx else "null"
_max_turns = max(4, ((effective_ctx or 4096) * 3 // 4) // 400 * 2)
_max_turns_js = str(_max_turns)

_STANDALONE_UI_HTML = r"""<!DOCTYPE html>
<html lang='ja'><head>
<meta charset='UTF-8'>
<meta name='viewport' content='width=device-width, initial-scale=1, maximum-scale=1, user-scalable=no'>
<title>Ollama Colab Private Chat — Standalone</title>
<link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;600&family=IBM+Plex+Sans+JP:wght@400;500&display=swap" rel="stylesheet">
<script src="https://cdnjs.cloudflare.com/ajax/libs/marked/9.1.6/marked.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/dompurify/3.0.8/purify.min.js"></script>
<style>
:root{--bg:#f5f0e8;--surface:#faf7f2;--border:#c8bfae;--accent:#3a6b4a;--text:#2c2820;--muted:#8a7f72;--user-bg:#e8edf5;--ai-bg:#f0f5ee}
*{box-sizing:border-box;margin:0;padding:0}
body{margin:0;background:var(--bg)}
#oc{font-family:'IBM Plex Sans JP',sans-serif;color:var(--text);max-width:1200px;width:100%;margin:0 auto;padding:clamp(12px,2vw,24px);min-height:100vh;display:flex;flex-direction:column}
.oc-ph{font-family:'IBM Plex Mono',monospace;display:flex;justify-content:space-between;align-items:baseline;padding:8px 0 12px;border-bottom:1px solid var(--border);margin-bottom:12px}
.oc-ph .t{font-size:13px;font-weight:600;letter-spacing:.04em}
.oc-ph .s{font-size:11px;color:var(--muted)}
.oc-sb{display:flex;align-items:center;gap:8px;margin-bottom:10px;font-size:12px;font-family:'IBM Plex Mono',monospace;color:var(--muted)}
.oc-sb input{flex:1;padding:5px 8px;border:1px solid var(--border);background:var(--surface);color:var(--text);font-family:'IBM Plex Mono',monospace;font-size:12px;outline:none}
.oc-card{border:1px solid var(--border);background:var(--surface);flex:1;display:flex;flex-direction:column}
.oc-ch{display:flex;justify-content:space-between;align-items:center;padding:8px 12px;border-bottom:1px solid var(--border);font-size:12px;font-family:'IBM Plex Mono',monospace}
.led{display:inline-block;width:8px;height:8px;border-radius:50%;background:#bbb;margin-right:6px;vertical-align:middle}
.led.on{background:var(--accent);animation:p 1.2s ease-in-out infinite}
@keyframes p{0%,100%{opacity:1}50%{opacity:.3}}
.tag{display:inline-block;padding:1px 6px;border:1px solid var(--border);font-size:11px;color:var(--muted);margin-left:4px}
.oc-log{flex:1;min-height:240px;overflow-y:auto;padding:12px;display:flex;flex-direction:column;gap:8px}
.sep{text-align:center;font-size:11px;color:var(--muted);font-family:'IBM Plex Mono',monospace;padding:4px 0;border-top:1px dashed var(--border);margin:4px 0}
.bbl{max-width:85%;padding:8px 12px;font-size:13px;line-height:1.65;word-break:break-word}
.bbl.u{align-self:flex-end;background:var(--user-bg);border:1px solid #c5d0e0}
.bbl.a{align-self:flex-start}
.bbl.e{align-self:flex-start;background:var(--ai-bg);border:1px solid #e0b8b8;color:#c0392b}
.bbl pre{background:#e8e4dc;padding:8px;overflow-x:auto;font-family:'IBM Plex Mono',monospace;font-size:12px;margin-top:6px;white-space:pre-wrap}
.tk{display:flex;gap:4px;padding:6px 0;align-items:center;min-height:20px}
.tk div{width:7px;height:7px;background:var(--accent);border-radius:50%;animation:tk 1.4s infinite ease-in-out;opacity:.4}
.tk div:nth-child(2){animation-delay:.2s}
.tk div:nth-child(3){animation-delay:.4s}
@keyframes tk{0%,80%,100%{transform:scale(0.8);opacity:.4}40%{transform:scale(1.2);opacity:1}}
.oc-ia{display:flex;gap:8px;padding:10px 12px;border-top:1px solid var(--border)}
.oc-ia textarea{flex:1;padding:7px 10px;border:1px solid var(--border);background:var(--bg);color:var(--text);font-family:'IBM Plex Sans JP',sans-serif;font-size:13px;resize:none;min-height:38px;max-height:120px;line-height:1.5;outline:none}
.oc-ia textarea::placeholder{font-size:clamp(10px,2.2vw,13px)}
.oc-ia button{padding:7px 18px;background:var(--accent);color:#fff;border:none;font-size:13px;cursor:pointer;font-family:'IBM Plex Mono',monospace;flex-shrink:0}
.oc-ia button:disabled,.oc-ia textarea:disabled,.disabled-btn{opacity:.5;pointer-events:none}
#ocAbortBtn{background:none;border:1px solid var(--border);color:var(--muted)}
#ocAbortBtn:hover{border-color:#c0392b;color:#c0392b}
.oc-cf{display:flex;justify-content:space-between;align-items:center;padding:6px 12px;border-top:1px solid var(--border);font-size:11px;font-family:'IBM Plex Mono',monospace;color:var(--muted)}
.oc-cf button{background:none;border:1px solid var(--border);padding:2px 8px;font-size:11px;color:var(--muted);cursor:pointer}
.bbl p{margin-bottom:8px}.bbl p:last-child{margin-bottom:0}.bbl h1,.bbl h2,.bbl h3{font-weight:600;margin:10px 0 4px}.bbl ul,.bbl ol{margin:6px 0 6px 18px}.bbl li{margin-bottom:2px}.bbl code{background:#e8e4dc;padding:1px 4px;font-family:'IBM Plex Mono',monospace;font-size:12px}.bbl blockquote{border-left:3px solid var(--border);padding-left:8px;color:var(--muted);margin:6px 0}.sep{cursor:pointer}.sep:hover{color:var(--accent)}.u-turn{display:flex;flex-direction:column;align-self:flex-end;gap:2px}.u-wrap{display:flex;align-items:flex-start;gap:6px}.u-foot{display:flex;justify-content:flex-end}.cp-btn{display:inline-block;margin:0;padding:3px 7px;font-size:13px;background:none;border:1px solid transparent;border-radius:6px;color:var(--muted);cursor:pointer;font-family:'IBM Plex Mono',monospace}.cp-btn:hover{background:rgba(0,0,0,.06);border-color:rgba(0,0,0,.15);color:var(--muted)}
.edit-ta{width:100%;min-height:60px;max-height:200px;padding:6px 8px;font-family:'IBM Plex Sans JP',sans-serif;font-size:13px;background:var(--bg);color:var(--text);border:1px solid var(--accent);resize:vertical;outline:none;box-sizing:border-box}
@media (max-width:600px){.oc-sb{display:none}}
#ocWebToggle{display:inline-flex;align-items:center;gap:6px;background:none;border:1px solid var(--border);padding:2px 8px 2px 6px;font-size:11px;color:var(--muted);cursor:pointer;font-family:'IBM Plex Mono',monospace;border-radius:10px;transition:background .15s,color .15s,border-color .15s}
#ocWebToggle.on{border-color:var(--accent);color:var(--accent)}
#ocWebToggle .sw-track{display:inline-block;width:24px;height:13px;border-radius:7px;background:var(--border);position:relative;flex-shrink:0;transition:background .15s}
#ocWebToggle .sw-track::after{content:'';position:absolute;top:2px;left:2px;width:9px;height:9px;border-radius:50%;background:#fff;transition:transform .15s}
#ocWebToggle.on .sw-track{background:var(--accent)}
#ocWebToggle.on .sw-track::after{transform:translateX(11px)}
.src-list{font-size:10px;color:var(--muted);font-family:'IBM Plex Mono',monospace;margin-top:4px;padding:4px 8px;border-left:2px solid var(--border);line-height:1.7}
.src-list a{color:var(--accent);text-decoration:none}
.src-list a:hover{color:var(--accent)}
</style>
</head>
<body>
<div id="oc">
  <div class="oc-ph">
    <span class="t">OLLAMA COLAB PRIVATE CHAT</span>
    <span class="s">no history · local inference</span>
  </div>
  <div class="oc-sb">
    BASE URL
    <input id="ocUrl" type="text" value="" readonly>
  </div>
  <div class="oc-card">
    <div class="oc-ch">
      <span><span id="ocLed" class="led"></span></span>
      <span><span class="tag" id="ocMTag"></span></span>
    </div>
    <div id="ocLog" class="oc-log"></div>
    <div class="oc-ia">
      <textarea id="ocIn" rows="1" placeholder="何でも聞いてみよう" autofocus></textarea>
      <button id="ocBtn" onclick="ocSend()">送信</button>
      <button id="ocAbortBtn" onclick="ocAbort()" style="display:none">中断</button>
    </div>
    <div class="oc-cf">
      <span>stateless · no localStorage</span>
      <div style="display:flex;gap:4px">
      <button id="ocWebToggle" title="Web検索RAGのON/OFF"><span class="sw-track"></span>🔍 Web検索</button>
      <button onclick="ocClear()">ログ消去</button>
    </div>
    </div>
  </div>
</div>
<script>
(function(){
const MODEL="__MODEL__", CTX=__CTX_JS__, MAX_TURNS=__MAX_TURNS__;
const $=id=>document.getElementById(id);
const elLog=$('ocLog'), elIn=$('ocIn'), elBtn=$('ocBtn'), elLed=$('ocLed'), elUrl=$('ocUrl');
const elAbortBtn = $('ocAbortBtn');
let _abort = null;
let _history = [];
window.ocAbort = function() { if (_abort) _abort(); };
const SEARXNG_ENABLED = __SEARXNG_ENABLED__;
let _webSearch = false;
const elWebToggle = $('ocWebToggle');
if (!SEARXNG_ENABLED) {
  elWebToggle.style.display = 'none';
} else {
  elWebToggle.onclick = function() {
    _webSearch = !_webSearch;
    elWebToggle.classList.toggle('on', _webSearch);
  };
}

let _lastSearchFailed = false;
async function fetchSearchResults(query) {
  _lastSearchFailed = false;
  try {
    const r = await fetch('/api/search?q=' + encodeURIComponent(query));
    if (!r.ok) { _lastSearchFailed = true; return []; }
    return await r.json();
  } catch(e) {
    console.warn('[RAG] fetchSearchResults failed:', e);
    _lastSearchFailed = true;
    return [];
  }
}
function renderSources(sources, targetEl) {
  if (!sources || !sources.length) return;
  const d = document.createElement('div');
  d.className = 'src-list';
  d.innerHTML = sources.map((s, i) =>
    `<sup><a href="${s.url}" target="_blank" rel="noopener" title="${s.url}">[${i+1}]</a></sup> ${s.title || s.url}`
  ).join('<br>');
  targetEl.appendChild(d);
}

elUrl.value = window.location.origin;
$('ocMTag').textContent = MODEL;

window.ocClear = function() { elLog.innerHTML = ''; _history = []; };
function ledOn(v) { elLed.className = 'led' + (v ? ' on' : ''); }
function md(s) { return DOMPurify.sanitize(marked.parse(s)); }
function addSep() {
  const d = document.createElement('div');
  d.className = 'sep';
  d.textContent = '↻ リセット';
  d.onclick = function(){
    while (elLog.firstChild && elLog.firstChild !== d)
      elLog.removeChild(elLog.firstChild);
    d.remove();
  };
  elLog.appendChild(d);
}
function addBubble(cls, content, asHtml) {
  const d = document.createElement('div'); d.className = 'bbl ' + cls;
  if (asHtml) d.innerHTML = content; else d.textContent = content;
  elLog.appendChild(d); return d;
}
function setDisabled(v) {
  v ? elBtn.classList.add('disabled-btn') : elBtn.classList.remove('disabled-btn');
  elAbortBtn.style.display = v ? '' : 'none';
}

window.ocSend = async function() {
  if (_abort) return;
  const t = elIn.value.trim();
  if (!t) return;
  elIn.value = ''; elIn.style.height = ''; elBtn.focus();
  const historyLenBefore = _history.length;
  _history.push({ role: 'user', content: t });
  let _sources = [];
  let _messagesForLLM = _history.slice(-MAX_TURNS);
  const _todayStr = new Date().toLocaleDateString('ja-JP', {year:'numeric',month:'long',day:'numeric',weekday:'long'});
  let _sysContent = `今日の日付: ${_todayStr}`;
  if (_webSearch) {
    const _searchInd = addBubble('a', '🔍 Web検索中…', false);
    elLog.scrollTop = elLog.scrollHeight;
    _sources = await fetchSearchResults(t);
    _searchInd.remove();
    if (_lastSearchFailed) {
      addBubble('a', '⚠️ Web検索に失敗しました。LLM単体で回答します。', false);
    }
    if (_sources.length) {
      const contextText = _sources.map((s, i) =>
        `[${i+1}] ${s.title}\n${s.content}`
      ).join('\n\n');
      _sysContent += `\n\nあなたは、提供されたWeb検索結果をもとに質問に回答するアシスタントです。\n\n【ルール】\n- 回答は検索結果を優先すること。引用する場合は出典番号 [n] を付けること。\n- 検索結果にない情報は「検索結果には含まれていません」と明示した上で、自身の知識で補足してよい。\n\n【検索結果】\n${contextText}`;
    }
  }
  _messagesForLLM = [{ role: 'system', content: _sysContent }, ..._messagesForLLM];
  if (elLog.children.length) addSep();
  const turnDiv = document.createElement('div');
  turnDiv.className = 'u-turn';
  const retryBtn = document.createElement('button');
  retryBtn.className = 'cp-btn'; retryBtn.textContent = '↻'; retryBtn.title = 'リトライ';
  function doRetry(newText) {
    if (_abort) return;
    const prev = turnDiv.previousSibling;
    if (prev && prev.classList.contains('sep')) prev.remove();
    while (turnDiv.nextSibling) elLog.removeChild(turnDiv.nextSibling);
    _history.length = historyLenBefore;
    turnDiv.remove();
    elIn.value = newText;
    ocSend();
  }
  retryBtn.onclick = function() { doRetry(t); };
  const userBubble = document.createElement('div');
  userBubble.className = 'bbl u';
  userBubble.textContent = t;
  const wrapDiv = document.createElement('div');
  wrapDiv.className = 'u-wrap';
  wrapDiv.appendChild(retryBtn);
  wrapDiv.appendChild(userBubble);
  const foot = document.createElement('div');
  foot.className = 'u-foot';
  const copyUserBtn = document.createElement('button');
  copyUserBtn.className = 'cp-btn'; copyUserBtn.textContent = '⧉'; copyUserBtn.title = 'コピー';
  copyUserBtn.onclick = function() { navigator.clipboard.writeText(t).then(function(){ copyUserBtn.textContent = '✓'; setTimeout(function(){ copyUserBtn.textContent = '⧉'; }, 1500); }); };
  foot.appendChild(copyUserBtn);

  const editBtn = document.createElement('button');
  editBtn.className = 'cp-btn'; editBtn.textContent = '✎'; editBtn.title = '編集';
  editBtn.onclick = enterEditMode;
  foot.appendChild(editBtn);

  function enterEditMode() {
    const ta = document.createElement('textarea');
    ta.className = 'edit-ta';
    ta.value = t;
    userBubble.replaceWith(ta);
    ta.focus(); ta.select();

    copyUserBtn.style.display = 'none';
    editBtn.style.display = 'none';

    const confirmBtn = document.createElement('button');
    confirmBtn.className = 'cp-btn'; confirmBtn.textContent = '✓'; confirmBtn.title = '確定';
    const cancelBtn = document.createElement('button');
    cancelBtn.className = 'cp-btn'; cancelBtn.textContent = '✕'; cancelBtn.title = 'キャンセル';

    function exitEditMode() {
      ta.replaceWith(userBubble);
      copyUserBtn.style.display = '';
      editBtn.style.display = '';
      confirmBtn.remove();
      cancelBtn.remove();
    }

    confirmBtn.onclick = function() {
      const newText = ta.value.trim();
      if (newText) doRetry(newText);
      else exitEditMode();
    };
    cancelBtn.onclick = exitEditMode;

    ta.addEventListener('keydown', function(e) {
      if (e.key === 'Enter' && (e.ctrlKey || e.metaKey)) { e.preventDefault(); confirmBtn.onclick(); }
    });

    foot.appendChild(confirmBtn);
    foot.appendChild(cancelBtn);
  }
  turnDiv.appendChild(wrapDiv);
  turnDiv.appendChild(foot);
  elLog.appendChild(turnDiv);
  elLog.scrollTop = elLog.scrollHeight;

  const ai = addBubble('a', '<div class="tk"><div></div><div></div><div></div></div>', true);
  setTimeout(function(){ ai.scrollIntoView({ block: 'start', behavior: 'instant' }); }, 0);
  setDisabled(true); ledOn(true);

  let full = '';
  try {
    const u = elUrl.value.replace(/[/]$/, '');
    const body = { model: MODEL, messages: _messagesForLLM, stream: true };
    if (CTX) body.options = { num_ctx: CTX };
    const ctrl = new AbortController();
    _abort = () => ctrl.abort();
    const res = await fetch(u + '/api/chat', {
      method: 'POST',
      headers: {
        'Content-Type': 'application/json'
      },
      body: JSON.stringify(body),
      signal: ctrl.signal
    });
    if (!res.ok) throw new Error('HTTP ' + res.status);
    const reader = res.body.getReader(), dec = new TextDecoder();
    let buf = '', firstToken = true;
    let _rafId = null;
    while (true) {
      const { done, value } = await reader.read();
      if (done) break;
      buf += dec.decode(value, { stream: true });
      const lines = buf.split('\n');
      buf = lines.pop();
      for (const l of lines) {
        if (!l.trim()) continue;
        try {
          const j = JSON.parse(l);
          const chunk = j.message?.content || '';
          if (!chunk) continue;
          if (firstToken) { ai.innerHTML = ''; firstToken = false; }
          full += chunk;
          if (_rafId === null) {
            _rafId = requestAnimationFrame(() => { ai.innerHTML = md(full); _rafId = null; });
          }
        } catch(_) {}
      }
    }
    if (full) {
      if (_rafId !== null) { cancelAnimationFrame(_rafId); _rafId = null; }
      ai.innerHTML = md(full);
      renderSources(_sources, ai);
      _history.push({ role: 'assistant', content: full });
      const btn = document.createElement('button');
      btn.className = 'cp-btn'; btn.textContent = '⧉'; btn.title = 'コピー';
      btn.onclick = function() { navigator.clipboard.writeText(full).then(function(){ btn.textContent = '✓'; setTimeout(function(){ btn.textContent = '⧉'; }, 1500); }); };
      ai.appendChild(btn);
    } else { ai.textContent = '（空の応答）'; }
  } catch(e) {
    _history.length = historyLenBefore;
    if (e.name === 'AbortError') {
      if (full) { ai.innerHTML = md(full); }
      else { ai.innerHTML = ''; ai.textContent = '（中断）'; }
    } else {
      ai.className = 'bbl e';
      ai.textContent = 'エラー: ' + e.message;
    }
  } finally {
    _abort = null;
    setDisabled(false); ledOn(false);
  }
};
elIn.addEventListener('keydown', e => {
  if (e.key === 'Enter' && (e.ctrlKey || e.metaKey)) { e.preventDefault(); ocSend(); return; }
  if (e.key === 'Enter' && !e.shiftKey && !e.isComposing) { e.preventDefault(); ocSend(); }
});
elIn.addEventListener('input', function() { this.style.height = ''; this.style.height = Math.min(this.scrollHeight, 120) + 'px'; });
elIn.focus();

// フォーム外のダブルタップでテキストエリアにフォーカス
let _lastClick = 0;
document.getElementById('oc').addEventListener('click', function(e) {
  let now = Date.now();
  if (now - _lastClick < 300 && !_abort && !e.target.closest('.oc-ia, .cp-btn, .edit-ta, button')) {
    elIn.focus();
  }
  _lastClick = now;
});
})();
</script>
</body></html>"""

# ── FastAPI アプリ ───────────────────────────────────────────────
_OLLAMA_B = "http://localhost:11434"
_share_app = FastAPI()

_sa_search_cache = TTLCache(maxsize=20, ttl=300)

async def _extract_search_query_async(user_text: str) -> str:
    """非同期コンテキスト用のクエリキーワード抽出。失敗時は元のテキストを返す。"""
    def _call_ollama():
        try:
            payload = {
                "model": _model,
                "prompt": (
                    "次のユーザー入力から、検索エンジン向けの短いキーワードを1行のみで抽出してください。"
                    "説明・前置き・記号は一切不要。キーワードだけを出力:\n" + user_text
                ),
                "stream": False,
                "options": {"num_ctx": 512},
            }
            import requests as _req
            r = _req.post(
                f"{_OLLAMA_B}/api/generate",
                json=payload,
                timeout=10,
            )
            keyword = r.json().get("response", "").strip()
            return keyword if keyword else user_text
        except Exception:
            return user_text
    return await asyncio.to_thread(_call_ollama)

@_share_app.get("/api/search")
async def _search_proxy(q: str):
    from fastapi.responses import JSONResponse
    query = await _extract_search_query_async(q)
    q_key = query.strip().lower()
    if q_key in _sa_search_cache:
        return JSONResponse(_sa_search_cache[q_key])
    try:
        def _do_search():
            with DDGS() as ddgs:
                return list(ddgs.text(
                    query=query,
                    region=SEARCH_REGION,
                    safesearch='moderate',
                    timelimit=_search_timelimit,
                    max_results=SEARCH_MAX_RESULTS,
                ) or [])
        results = await asyncio.to_thread(_do_search)
        formatted = [
            {
                "title":   r.get("title", ""),
                "url":     r.get("href", ""),
                "content": r.get("body", "")[:SEARCH_BODY_LENGTH],
            }
            for r in results
        ]
        _sa_search_cache[q_key] = formatted
        return JSONResponse(formatted)
    except Exception as e:
        print(f"DuckDuckGo Search Error: {e}")
        return JSONResponse([])

@_share_app.get("/")
async def _index_b():
    return HTMLResponse(_html_b)

@_share_app.post("/api/{path:path}")
async def _proxy_b(path: str, request: Request):
    body = await request.body()
    async def _stream():
        async with httpx.AsyncClient(timeout=300) as _c:
            async with _c.stream(
                "POST", f"{_OLLAMA_B}/api/{path}",
                content=body,
                headers={"Content-Type": "application/json"},
            ) as _r:
                async for _chunk in _r.aiter_bytes():
                    yield _chunk
    return StreamingResponse(_stream(), media_type="application/x-ndjson")

# ── HTML 生成（__TUNNEL_URL__ replace は不要） ───────────────────
_html_b = (
    _STANDALONE_UI_HTML
    .replace("__MODEL__",     _model)
    .replace("__CTX_JS__",    _ctx_js)
    .replace("__MAX_TURNS__", _max_turns_js)
    .replace("__SEARXNG_ENABLED__",  SEARXNG_ENABLED_JS)
)

# ── uvicorn 起動 ─────────────────────────────────────────────────
print(f"  {BLUE}◦{RESET} Server   FastAPI を起動中 (port 11435)")
threading.Thread(
    target=lambda: uvicorn.run(_share_app, host="0.0.0.0", port=11435, log_level="error"),
    daemon=True,
).start()

for _ in range(30):
    try:
        if requests.get("http://localhost:11435/", timeout=1).status_code == 200:
            print(f"  {GREEN}✓{RESET} Server   FastAPI 起動完了")
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(0.3)
else:
    raise RuntimeError("⚠️ FastAPI の起動確認に失敗しました。")

# ── Cloudflare Tunnel (port 11435) ───────────────────────────────
print(f"  {BLUE}◦{RESET} Network  Cloudflare Tunnel 確立中 (port 11435)")
_cf_b = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11435"],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE,
)
_url_b = None
for _line in iter(_cf_b.stderr.readline, b""):
    _m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", _line.decode())
    if _m:
        _url_b = _m.group(0)
        break
if not _url_b:
    raise RuntimeError("⚠️ Cloudflare Tunnel の URL 取得に失敗しました。")

# ── DNS 疎通確認 ──────────────────────────────────────────────────
_DNS_RETRIES  = 20
_DNS_INTERVAL = 3
print(f"  {BLUE}◦{RESET} Network  DNS 伝播を確認中", end="", flush=True)
for _ in range(_DNS_RETRIES):
    try:
        _r = requests.get(_url_b, timeout=4, allow_redirects=False)
        if _r.status_code < 500:
            print(f"\n  {GREEN}✓{RESET} Network  DNS 解決・到達確認完了")
            break
    except requests.exceptions.RequestException:
        pass
    print(".", end="", flush=True)
    time.sleep(_DNS_INTERVAL)
else:
    print(f"\n  ⚠️ DNS 確認タイムアウト。URL は表示しますが、アクセスに数秒かかる場合があります。")
print(f"  {GREEN}✓{RESET} Network  Tunnel 確立完了")

# ── 出力 ─────────────────────────────────────────────────────────
print(f"\n{GRAY}──────────────────────────────────────────{RESET}")
print(f"Standalone URL : {GREEN}{_url_b}{RESET}")
print(f"{GRAY}──────────────────────────────────────────{RESET}")

display(HTML(
    f'<div style="display:inline-flex;flex-direction:column;border:1px solid #c8bfae;'
    f'background:#faf7f2;color:#2c2820;font-family:monospace;font-size:11px;margin-top:4px">'
    f'<div style="padding:6px 12px;border-bottom:1px solid #c8bfae;display:flex;align-items:center;gap:8px">'
    f'<span style="width:7px;height:7px;border-radius:50%;background:#3a6b4a;display:inline-block;flex-shrink:0"></span>'
    f'<span style="letter-spacing:.04em">&#x2713; Chat UI &mdash; Standalone</span></div>'
    f'<a href="{_url_b}" target="_blank" style="text-decoration:none;color:#2c2820;'
    f'display:flex;align-items:center;justify-content:space-between;padding:8px 12px;gap:24px">'
    f'<span style="overflow:hidden;text-overflow:ellipsis;white-space:nowrap">{_url_b}</span>'
    f'<span style="flex-shrink:0">&#x2197;</span></a></div>'
))